In [42]:
import pandas as pd
import numpy as np

In [43]:
#Membuat dataset baru, berupa data Mahasiswa angkatan 22
np.random.seed(42)
#Jumlah mahasiswa yang dibutuhkan
jumlah_mahasiswa = 300

In [44]:
#Tahapan 1: Mengidentifikasi feature/indikator apa saja yang diperlukan
#Membuat korelasi antara MK Gagal dan SKS Lulus
jumlah_mk_gagal_array = np.random.randint(0, 5, jumlah_mahasiswa)
sks_ideal = np.random.randint(135, 140, jumlah_mahasiswa)
sks_lulus_array = sks_ideal - (jumlah_mk_gagal_array * 3)
#Membagi peluang Status Skripsi 
status_list = []
bimbingan_list = []

for sks in sks_lulus_array:
    if sks < 120:
        # SKS kurang, pasti belum skripsi
        status = 'Belum'
        bimbingan = np.random.randint(0, 3)
    elif 120 <= sks < 125:
        # Mulai mengajukan judul
        status = np.random.choice(['Belum', 'ACC Judul'])
        bimbingan = np.random.randint(2, 6) if status == 'ACC Judul' else np.random.randint(0, 3)
    elif 125 <= sks < 130:
        # Tahap penyusunan proposal
        status = np.random.choice(['ACC Judul', 'Sempro'])
        bimbingan = np.random.randint(5, 9)
    elif 130 <= sks < 135:
        status = np.random.choice(['Sempro', 'Semhas'])
        # PERBAIKAN: Jika sudah Semhas, bimbingan diset minimal 11 agar PASTI lolos rule (>10)
        bimbingan = np.random.randint(11, 15) if status == 'Semhas' else np.random.randint(7, 11)
    else: 
        # SKS >= 135, sudah tahap akhir (Semhas/Skripsi)
        status = np.random.choice(['Semhas', 'Skripsi'])
        # PERBAIKAN: Pastikan bimbingan di atas 10 agar lolos rule ontime
        bimbingan = np.random.randint(11, 21)
        
    status_list.append(status)
    bimbingan_list.append(bimbingan)
#Membuat data
data = {
    'NIM': [f"22021{str(i).zfill(3)}" for i in range(1, jumlah_mahasiswa + 1)],
    'Tahun_Masuk': np.full(jumlah_mahasiswa, 2022),
    'IPS_Semester_1': np.round(np.random.uniform(2.5, 4.0, jumlah_mahasiswa), 2),
    'IPS_Semester_2': np.round(np.random.uniform(2.5, 4.0, jumlah_mahasiswa), 2),
    'IPS_Semester_3': np.round(np.random.uniform(2.5, 4.0, jumlah_mahasiswa), 2),
    'IPS_Semester_4': np.round(np.random.uniform(2.5, 4.0, jumlah_mahasiswa), 2),
    'IPS_Semester_5': np.round(np.random.uniform(2.5, 4.0, jumlah_mahasiswa), 2),
    'IPS_Semester_6': np.round(np.random.uniform(2.5, 4.0, jumlah_mahasiswa), 2),
    'IPS_Semester_7': np.round(np.random.uniform(2.5, 4.0, jumlah_mahasiswa), 2),
    'Jumlah_MK_Gagal': jumlah_mk_gagal_array,
    'SKS_Lulus': sks_lulus_array,
    'Status_Skripsi': status_list,
    'Jumlah_Bimbingan': bimbingan_list
}
df = pd.DataFrame(data)


In [45]:
#Tahapan Kedua: Analisis distribusi data (Rule-Based System)
rata_IPK = df[['IPS_Semester_1', 'IPS_Semester_2', 'IPS_Semester_3', 'IPS_Semester_4', 'IPS_Semester_5', 'IPS_Semester_6', 'IPS_Semester_7']].mean(axis=1)
dict_status = {'Belum': 0, 'ACC Judul': 1, 'Sempro': 2, 'Semhas': 3, 'Skripsi': 4}
status = df['Status_Skripsi'].map(dict_status)

#Rule Kelulusan
ontime = (
    (rata_IPK >= 3.00) &
    (df['Jumlah_MK_Gagal'] <= 2) &
    (df['Jumlah_Bimbingan'] > 10) &
    (status >= 3)
)

#Jika mahasiswa memenuhi "kondisi_ontime", maka diberi label 1 (Ontime / <= 4 Tahun)
#Jika TIDAK memenuhi, maka diberi label 0 (Telat / > 4 Tahun)
df['Ontime'] = np.where(ontime, 1, 0) 
#Mengacak urutan baris DataFrame secara total
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [46]:
#Tahapan Ketiga: Export ke file CSV
nama_file = 'Dataset_Mahasiswa1_2022.csv'
df.to_csv(nama_file, index=False)

print(f"Dataset berhasil dibuat dan disimpan sebagai '{nama_file}'.")
print(df[['NIM', 'SKS_Lulus', 'Jumlah_MK_Gagal', 'Status_Skripsi', 'Jumlah_Bimbingan', 'Ontime']].head(15))

Dataset berhasil dibuat dan disimpan sebagai 'Dataset_Mahasiswa1_2022.csv'.
         NIM  SKS_Lulus  Jumlah_MK_Gagal Status_Skripsi  Jumlah_Bimbingan  \
0   22021204        137                0        Skripsi                14   
1   22021267        128                3         Sempro                 6   
2   22021153        139                0         Semhas                19   
3   22021010        127                4         Sempro                 7   
4   22021234        130                3         Semhas                12   
5   22021227        136                1        Skripsi                18   
6   22021197        131                2         Semhas                13   
7   22021110        132                2         Semhas                13   
8   22021006        135                1         Semhas                11   
9   22021176        136                1        Skripsi                16   
10  22021238        128                3         Sempro                 8   
